In [1]:
import os, re, json, textwrap
from pathlib import Path

import numpy as np
import requests
import chromadb

# Progress bar for the one-time embedding step (optional).
try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

In [2]:
OLLAMA_URL  = "http://localhost:11434"
CHAT_MODEL  = "llama3.1:8b"
EMBED_MODEL = "embeddinggemma:latest"        # e.g. "nomic-embed-text"

WORDS_PER_CHUNK = 300      # size of each chunk, in words
OVERLAP_WORDS   = 60       # chunks overlap so we don't slice a thought in half
TOPK            = 5        # how many chunks to retrieve per question

CORPUS_DIR        = Path("corpus_jupyter")   # downloaded .txt books land here
CHROMA_PATH       = "./chroma_db"            # Chroma persists vectors here
CHROMA_COLLECTION = "rag_demo"

EMBED_BATCH_SIZE  = 64
DOWNLOAD_FROM_WEB = True    # False = stay offline, use whatever .txt is already in CORPUS_DIR

In [3]:
GUTENBERG_BOOKS = {
    "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",
}
print(f"{len(GUTENBERG_BOOKS)} books in the corpus.")

13 books in the corpus.


# Talk to Ollama

In [4]:
SESSION = requests.Session()
SESSION.trust_env = False

# Sanity check: is Ollama up, and are our two models pulled?
r = SESSION.get(f"{OLLAMA_URL}/api/tags", timeout=30)
r.raise_for_status()
available = [m.get("name", "") for m in r.json().get("models", [])]
print(f"Ollama reachable — {len(available)} model(s) installed.")
for name in (CHAT_MODEL, EMBED_MODEL):
    print(f"  {name}: {'ok' if name in available else 'MISSING — run `ollama pull`'}")

Ollama reachable — 12 model(s) installed.
  llama3.1:8b: ok
  embeddinggemma:latest: ok


In [5]:
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

# Project Gutenberg wraps each book in license boilerplate; keep only what's between these.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

docs = []
for title, url in GUTENBERG_BOOKS.items():
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")    # filesystem-safe filename
    path = CORPUS_DIR / f"{safe_name}.txt"

    if not path.exists():
        if not DOWNLOAD_FROM_WEB:
            continue
        print(f"Downloading: {title}")
        resp = SESSION.get(url, timeout=180)
        resp.raise_for_status()
        raw = resp.text
        start, end = START_MARK.search(raw), END_MARK.search(raw)
        if start and end and end.start() > start.end():
            raw = raw[start.end():end.start()]
        path.write_text(raw.strip(), encoding="utf-8")

    docs.append({"title": title, "url": url,
                 "text": path.read_text(encoding="utf-8", errors="ignore")})

print(f"Loaded {len(docs)} books.")

Loaded 13 books.


In [6]:
chunks = []
for d in docs:
    doc_id = re.sub(r"[^a-z0-9]+", "-", d["title"].lower()).strip("-")   # "pride-and-prejudice"

    words = re.sub(r"\s+", " ", d["text"]).strip().split()
    step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)

    i = 0
    for start in range(0, len(words), step):
        window = words[start:start + WORDS_PER_CHUNK]
        if len(window) < max(60, WORDS_PER_CHUNK // 4):    # drop a tiny trailing scrap
            break
        chunks.append({
            "id": f"{doc_id}#{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "title": d["title"],
            "source": d["url"],
            "text": " ".join(window),
        })
        i += 1
        if start + WORDS_PER_CHUNK >= len(words):
            break

print(f"Built {len(chunks)} chunks from {len(docs)} books.\n")
print("Example chunk:")
print(f"  id   : {chunks[0]['id']}")
print(f"  words: {len(chunks[0]['text'].split())}")
print("  text :", textwrap.fill(chunks[0]['text'][:300] + " ...", width=80,
                                subsequent_indent="         "))

Built 8509 chunks from 13 books.

Example chunk:
  id   : moby-dick#0
  words: 300
  text : MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS
         (Supplied by a Sub-Sub-Librarian). CHAPTER 1. Loomings. CHAPTER 2. The
         Carpet-Bag. CHAPTER 3. The Spouter-Inn. CHAPTER 4. The Counterpane.
         CHAPTER 5. Breakfast. CHAPTER 6. The Street. CHAPTER 7. The Chapel.
         CHAPTER 8. The Pulp ...


In [7]:
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    CHROMA_COLLECTION, metadata={"hnsw:space": "cosine"}   # cosine distance: lower = closer
)

already_built = collection.count() > 0
print(f"Chroma collection '{CHROMA_COLLECTION}': {collection.count()} vectors.")
print("Vectors already present — the embed step will be skipped." if already_built
      else "Empty collection — we'll embed and populate it below.")

Chroma collection 'rag_demo': 8509 vectors.
Vectors already present — the embed step will be skipped.


In [8]:
if already_built:
    print(f"Skipping embed — Chroma already has {collection.count()} vectors.")
else:
    batches = range(0, len(chunks), EMBED_BATCH_SIZE)
    if tqdm:
        batches = tqdm(batches, desc=f"Embedding with {EMBED_MODEL}")

    for s in batches:
        batch = chunks[s:s + EMBED_BATCH_SIZE]
        texts = [c["text"] for c in batch]

        # One POST returns one vector per text.
        resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                            json={"model": EMBED_MODEL, "input": texts}, timeout=600)
        resp.raise_for_status()
        vectors = resp.json()["embeddings"]

        collection.add(
            ids=[c["id"] for c in batch],
            documents=texts,
            metadatas=[{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"],
                        "title": c["title"], "source": c["source"]} for c in batch],
            embeddings=vectors,
        )

    print(f"Done. Chroma now holds {collection.count()} vectors.")

Skipping embed — Chroma already has 8509 vectors.


In [9]:
question = "Who is Elizabeth Bennet?"

# Embed the question (same endpoint as before, just one text).
resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                    json={"model": EMBED_MODEL, "input": [question]}, timeout=600)
resp.raise_for_status()
query_vector = resp.json()["embeddings"][0]

# Ask Chroma for the nearest chunks. We sent one query, so results live at index [0].
res = collection.query(query_embeddings=[query_vector], n_results=TOPK,
                       include=["documents", "metadatas", "distances"])
docs_out  = res["documents"][0]
metas_out = res["metadatas"][0]
dists_out = res["distances"][0]

print(f"Top {TOPK} chunks for: {question!r}\n")
for meta, dist in zip(metas_out, dists_out):
    print(f"  distance={dist:.4f}  [{meta['doc_id']}#{meta['chunk_index']}]  {meta['title']}")

Top 5 chunks for: 'Who is Elizabeth Bennet?'

  distance=0.4757  [pride-and-prejudice#23]  Pride and Prejudice
  distance=0.5044  [pride-and-prejudice#490]  Pride and Prejudice
  distance=0.5237  [pride-and-prejudice#482]  Pride and Prejudice
  distance=0.5250  [pride-and-prejudice#486]  Pride and Prejudice
  distance=0.5263  [pride-and-prejudice#34]  Pride and Prejudice


In [13]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

In [14]:
question = "How does Victor Frankenstein create life?"

# --- Retrieve: embed the question, find the nearest chunks --------------------
resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                    json={"model": EMBED_MODEL, "input": [question]}, timeout=600)
resp.raise_for_status()
query_vector = resp.json()["embeddings"][0]

res = collection.query(query_embeddings=[query_vector], n_results=TOPK,
                       include=["documents", "metadatas", "distances"])
hits = list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

# --- Build the context block: a [doc_id#chunk] tag before each chunk ----------
context = ""
for text, meta, dist in hits:
    context += f"[{meta['doc_id']}#{meta['chunk_index']}] {meta['title']}\n{text}\n---\n"

# --- Ask the LLM to answer ONLY from that context -----------------------------
system_prompt = (
    "You are a helpful assistant. Answer ONLY using the provided context. "
    "If the answer is not in the context, say: 'I don't know based on the provided context.' "
    "Cite sources in square brackets like [doc_id#chunk_index] for each key claim."
)
user_prompt = f"Question: {question}\n\nContext:\n{context}"

print("The context for the question, built from the retrieved chunks, looks like this:\n")
pretty_print(context)

print("Asking the LLM to answer the question based on the retrieved context...\n")

resp = SESSION.post(f"{OLLAMA_URL}/api/chat",
                    json={"model": CHAT_MODEL, "stream": False,
                          "options": {"temperature": 0.2},
                          "messages": [{"role": "system", "content": system_prompt},
                                       {"role": "user", "content": user_prompt}]},
                    timeout=600)
resp.raise_for_status()
answer = resp.json()["message"]["content"]

# --- Show it ------------------------------------------------------------------
print("[Q]", question, "\n")
print("[ANSWER]\n")
pretty_print(answer.strip())
print("\n[SOURCES RETRIEVED]")
for text, meta, dist in hits:
    print(f"  distance={dist:.4f}  [{meta['doc_id']}#{meta['chunk_index']}]  {meta['title']}")

The context for the question, built from the retrieved chunks, looks like this:

[frankenstein#62] Frankenstein it breathed hard, and a convulsive motion
agitated its limbs. How can I describe my emotions at this catastrophe, or how
delineate the wretch whom with such infinite pains and care I had endeavoured to
form? His limbs were in proportion, and I had selected his features as
beautiful. Beautiful! Great God! His yellow skin scarcely covered the work of
muscles and arteries beneath; his hair was of a lustrous black, and flowing; his
teeth of a pearly whiteness; but these luxuriances only formed a more horrid
contrast with his watery eyes, that seemed almost of the same colour as the dun-
white sockets in which they were set, his shrivelled complexion and straight
black lips. The different accidents of life are not so changeable as the
feelings of human nature. I had worked hard for nearly two years, for the sole
purpose of infusing life into an inanimate body. For this I had depri